In [ ]:
"""
Expense Tracker (with user input) - Python + NumPy + Pandas
Add expenses, view monthly summary, category-wise spending, budget alerts.
Data is saved in expenses.csv
"""
import os
from datetime import datetime
import numpy as np
import pandas as pd

FILE = "expenses.csv"
CATEGORIES = ["Food", "Travel", "Rent", "Shopping", "Bills", "Health", "Education", "Other"]


def load():
    if os.path.exists(FILE) and os.path.getsize(FILE) > 0:
        df = pd.read_csv(FILE)
        df["date"] = pd.to_datetime(df["date"])
        return df
    return pd.DataFrame(columns=["date", "category", "amount", "note"])


def save(df):
    df.to_csv(FILE, index=False)


def ask_float(prompt):
    while True:
        try:
            v = float(input(prompt))
            if v <= 0:
                print("  Amount must be greater than 0")
                continue
            return v
        except ValueError:
            print("  Enter a valid number")


def ask_date(prompt):
    while True:
        s = input(prompt).strip()
        if s == "":
            return datetime.today()
        try:
            return datetime.strptime(s, "%Y-%m-%d")
        except ValueError:
            print("  Use format YYYY-MM-DD (or press Enter for today)")


def add_expense():
    df = load()
    print("\nCategories:")
    for i, c in enumerate(CATEGORIES, 1):
        print(f"  {i}. {c}")
    while True:
        try:
            k = int(input("Choose category number: "))
            if 1 <= k <= len(CATEGORIES):
                break
        except ValueError:
            pass
        print("  Invalid choice")
    amount = ask_float("Amount (Rs): ")
    date = ask_date("Date YYYY-MM-DD (Enter = today): ")
    note = input("Note (optional): ").strip()
    row = pd.DataFrame([[date, CATEGORIES[k - 1], amount, note]], columns=df.columns)
    df = pd.concat([df, row], ignore_index=True)
    save(df)
    print(f"  Saved: Rs {amount:,.0f} on {CATEGORIES[k - 1]}")


def view_all():
    df = load()
    if df.empty:
        print("\nNo expenses yet.")
        return
    out = df.sort_values("date", ascending=False).reset_index(drop=True)
    out["date"] = out["date"].dt.strftime("%d-%b-%Y")
    print("\n" + out.to_string())


def monthly_summary():
    df = load()
    if df.empty:
        print("\nNo expenses yet.")
        return
    df["month"] = df["date"].dt.to_period("M")
    m = df.groupby("month")["amount"].agg(["sum", "count", "mean", "max"]).round(0)
    m.columns = ["Total", "Entries", "Average", "Biggest"]
    print("\n--- Monthly Summary ---")
    print(m.to_string())
    print(f"\nOverall spent   : Rs {df.amount.sum():,.0f}")
    print(f"Monthly average : Rs {m.Total.mean():,.0f}")


def category_report():
    df = load()
    if df.empty:
        print("\nNo expenses yet.")
        return
    c = df.groupby("category")["amount"].sum().sort_values(ascending=False)
    pct = np.round(c / c.sum() * 100, 1)
    rep = pd.DataFrame({"Spent": c, "Percent": pct})
    print("\n--- Category-wise Spending ---")
    print(rep.to_string())
    print(f"\nTop category: {rep.index[0]} ({rep.Percent.iloc[0]}%)")


def budget_check():
    df = load()
    budget = ask_float("\nEnter monthly budget (Rs): ")
    now = datetime.today()
    this = df[(df.date.dt.month == now.month) & (df.date.dt.year == now.year)]
    spent = this.amount.sum()
    left = budget - spent
    print(f"\nSpent this month : Rs {spent:,.0f}")
    print(f"Budget           : Rs {budget:,.0f}")
    if left >= 0:
        print(f"Remaining        : Rs {left:,.0f} ({left / budget * 100:.0f}% left)")
        if spent / budget >= 0.8:
            print("WARNING: You have used over 80% of your budget!")
    else:
        print(f"OVER BUDGET by Rs {abs(left):,.0f}!")


def delete_last():
    df = load()
    if df.empty:
        print("\nNothing to delete.")
        return
    last = df.iloc[-1]
    if input(f"\nDelete Rs {last.amount:,.0f} ({last.category})? (y/n): ").lower() == "y":
        save(df.iloc[:-1])
        print("  Deleted.")


def main():
    menu = {"1": ("Add expense", add_expense), "2": ("View all expenses", view_all),
            "3": ("Monthly summary", monthly_summary), "4": ("Category report", category_report),
            "5": ("Budget check", budget_check), "6": ("Delete last entry", delete_last)}
    while True:
        print("\n===== EXPENSE TRACKER =====")
        for k, (name, _) in menu.items():
            print(f"{k}. {name}")
        print("0. Exit")
        choice = input("Choose: ").strip()
        if choice == "0":
            print("Goodbye!")
            break
        if choice in menu:
            menu[choice][1]()
        else:
            print("Invalid choice")


if __name__ == "__main__":
    main()



===== EXPENSE TRACKER =====
1. Add expense
2. View all expenses
3. Monthly summary
4. Category report
5. Budget check
6. Delete last entry
0. Exit


ASEEM PRATAP SINGH
AIML-2
2026B15310059